In [12]:
# %%
import pandas as pd
import numpy as np
import requests
import zipfile
import io
import re
from datetime import datetime, timedelta

# =========================================
# CONFIG
# =========================================

UNDERLYING_MAP = {
    "PETR": "PETR4",
    "VALE": "VALE3",
    "BBDC": "BBDC4",
    "BBAS": "BBAS3",
    "ABEV": "ABEV3",
    "WEGE": "WEGE3",
    "B3SA": "B3SA3",
    "SUZB": "SUZB3",
}

UNDERLYINGS_BASE = list(UNDERLYING_MAP.keys())

# ticker válido de opção B3
OPTION_REGEX = re.compile(r"^[A-Z]{4}[A-Z][0-9]{2,3}$")


# =========================================
# FUNÇÃO: BAIXAR ÚLTIMO COTAHIST DISPONÍVEL
# =========================================

def download_latest_cotahist(max_lookback_days=10):

    base_url = "https://bvmf.bmfbovespa.com.br/InstDados/SerHist/COTAHIST_D{date}.ZIP"
    today = datetime.today()

    for i in range(max_lookback_days):

        ref = today - timedelta(days=i)
        date_str = ref.strftime("%d%m%Y")
        url = base_url.format(date=date_str)

        try:

            r = requests.get(url, timeout=20)

            if r.status_code != 200:
                continue

            z = zipfile.ZipFile(io.BytesIO(r.content))
            file_name = z.namelist()[0]
            data = z.read(file_name).decode("latin1")
            lines = data.splitlines()

            print("✅ Arquivo encontrado:", url)
            print("Arquivo interno:", file_name)
            print("Total de linhas:", len(lines))

            return ref.date(), lines

        except Exception:
            continue

    raise RuntimeError("❌ Não foi possível baixar um COTAHIST recente.")


# =========================================
# FUNÇÃO: PARSE DO COTAHIST
# =========================================

def parse_cotahist(lines, underlyings_base):

    records = []

    for l in lines:

        if not l.startswith("01"):
            continue

        ticker = l[12:24].strip().upper()

        if not any(ticker.startswith(u) for u in underlyings_base):
            continue

        close = int(l[108:121]) / 100
        volume = int(l[152:170])

        strike_raw = l[188:201]
        expiration_raw = l[202:210]

        strike = np.nan
        expiration = pd.NaT


        # =====================================
        # STRIKE
        # =====================================

        if strike_raw.strip().isdigit():

            strike_val = int(strike_raw)

            if strike_val != 0:

                strike = strike_val / 100

                # REMOVE STRIKES TERMINADOS EM .0
                # elimina PETRD540, PETRD550 etc
                if strike.is_integer():
                    strike = np.nan


        # =====================================
        # EXPIRATION
        # =====================================

        if expiration_raw.strip().isdigit():

            exp_val = int(expiration_raw)

            if exp_val != 0:
                expiration = datetime.strptime(expiration_raw, "%Y%m%d")


        # =====================================
        # FILTRO DE TICKER REAL DE OPÇÃO
        # =====================================

        if pd.notna(strike):

            # elimina séries fantasmas tipo:
            # PETRD540W5
            # PETRD540M
            # PETRD540T
            # etc

            if not OPTION_REGEX.match(ticker):
                continue


        records.append({

            "ticker": ticker,
            "fechamento_d-1": close,
            "volume_d-1": volume,
            "strike": strike,
            "expiration": expiration

        })

    return pd.DataFrame(records)

# =========================================
# EXECUÇÃO
# =========================================

trade_date, lines = download_latest_cotahist()

df_raw = parse_cotahist(lines, UNDERLYINGS_BASE)

print("Data de referência:", trade_date)
print("Qtd registros após filtro base:", len(df_raw))

display(df_raw.head(20))

✅ Arquivo encontrado: https://bvmf.bmfbovespa.com.br/InstDados/SerHist/COTAHIST_D13032026.ZIP
Arquivo interno: COTAHIST_D13032026.TXT
Total de linhas: 17472
Data de referência: 2026-03-13
Qtd registros após filtro base: 2972


,ticker,fechamento_d-1,volume_d-1,strike,expiration
0,VALE3,78.30,14413500,NaN,9999-12-31
1,SUZB3,53.40,3469600,NaN,9999-12-31
2,BBAS3,23.81,38119800,NaN,9999-12-31
3,PETR3,49.38,11663100,NaN,9999-12-31
4,PETR4,44.67,42518400,NaN,9999-12-31
5,B3SA3,16.88,19621800,NaN,9999-12-31
6,BBDC3,16.42,5190900,NaN,9999-12-31
7,BBDC4,18.99,33704500,NaN,9999-12-31
8,ABEV3,15.00,25418200,NaN,9999-12-31
9,WEGE3,46.21,4406200,NaN,9999-12-31


In [13]:
# %%
from datetime import datetime, timedelta

# =========================================
# 3ª SEXTA
# =========================================

def third_friday(year, month):
    d = datetime(year, month, 1)
    days_until_friday = (4 - d.weekday()) % 7
    first_friday = d + timedelta(days=days_until_friday)
    third = first_friday + timedelta(days=14)
    return third.date()


# usa a data do arquivo, não o relógio do PC
now = datetime.combine(trade_date, datetime.min.time())

current_expiry_date = third_friday(now.year, now.month)

if now.month == 12:
    next_month = 1
    next_year = now.year + 1
else:
    next_month = now.month + 1
    next_year = now.year

next_expiry_date = third_friday(next_year, next_month)

print("Vencimento mês atual :", current_expiry_date)
print("Vencimento próximo mês:", next_expiry_date)


# =========================================
# CLASSIFICAÇÃO
# =========================================

def classify_asset(ticker, strike):
    if pd.notna(strike):
        letter = ticker[4]

        if letter in list("ABCDEFGHIJKL"):
            return "CALL"
        elif letter in list("MNOPQRSTUVWX"):
            return "PUT"
        else:
            return "OPTION"

    return "SPOT"


df_raw["type"] = df_raw.apply(
    lambda r: classify_asset(r["ticker"], r["strike"]),
    axis=1
)


# =========================================
# UNDERLYING CORRETO
# =========================================

def infer_underlying(ticker, asset_type):
    base = ticker[:4]

    if asset_type == "SPOT":
        return ticker

    return UNDERLYING_MAP.get(base)


df_raw["underlying"] = df_raw.apply(
    lambda r: infer_underlying(r["ticker"], r["type"]),
    axis=1
)

print(df_raw["type"].value_counts(dropna=False))
display(df_raw.head(20))

Vencimento mês atual : 2026-03-20
Vencimento próximo mês: 2026-04-17
type
CALL    1440
PUT     1145
SPOT     387
Name: count, dtype: int64


,ticker,fechamento_d-1,volume_d-1,strike,expiration,type,underlying
0,VALE3,78.30,14413500,NaN,9999-12-31,SPOT,VALE3
1,SUZB3,53.40,3469600,NaN,9999-12-31,SPOT,SUZB3
2,BBAS3,23.81,38119800,NaN,9999-12-31,SPOT,BBAS3
3,PETR3,49.38,11663100,NaN,9999-12-31,SPOT,PETR3
4,PETR4,44.67,42518400,NaN,9999-12-31,SPOT,PETR4
5,B3SA3,16.88,19621800,NaN,9999-12-31,SPOT,B3SA3
6,BBDC3,16.42,5190900,NaN,9999-12-31,SPOT,BBDC3
7,BBDC4,18.99,33704500,NaN,9999-12-31,SPOT,BBDC4
8,ABEV3,15.00,25418200,NaN,9999-12-31,SPOT,ABEV3
9,WEGE3,46.21,4406200,NaN,9999-12-31,SPOT,WEGE3


In [14]:
# %%
# =========================================
# UNIVERSO IGUAL AO MT5
# =========================================

df_spot = df_raw[df_raw["type"] == "SPOT"].copy()

df_calls = df_raw[
    (df_raw["type"] == "CALL") &
    (df_raw["expiration"].notna()) &
    (df_raw["expiration"].dt.date.isin([current_expiry_date, next_expiry_date]))
].copy()

print("SPOT:", len(df_spot))
print("CALL:", len(df_calls))


# =========================================
# PREÇO DO SPOT NAS CALLs
# =========================================

spot_price_map = dict(zip(df_spot["ticker"], df_spot["fechamento_d-1"]))
df_calls["spot_price"] = df_calls["underlying"].map(spot_price_map)

print("CALLs com spot mapeado:", df_calls["spot_price"].notna().sum())
display(df_calls.head(20))


# =========================================
# CALLs OTM
# =========================================

df_calls_valid = df_calls[
    (df_calls["strike"].notna()) &
    (df_calls["spot_price"].notna()) &
    (df_calls["strike"] > df_calls["spot_price"])
].copy()

print("CALLs OTM:", len(df_calls_valid))

display(
    df_calls_valid[
        ["ticker", "underlying", "spot_price", "strike", "expiration", "fechamento_d-1", "volume_d-1"]
    ].head(20)
)


# =========================================
# GERAR CALL CREDIT SPREADS
# =========================================

spreads = []

group_cols = ["underlying", "expiration"]

for (underlying, expiration), g in df_calls_valid.groupby(group_cols):

    g = g.sort_values("strike")

    for _, sell in g.iterrows():

        for _, buy in g.iterrows():

            if buy["strike"] <= sell["strike"]:
                continue

            credit = sell["fechamento_d-1"] - buy["fechamento_d-1"]

            if credit <= 0:
                continue

            spreads.append({

                "underlying": underlying,
                "expiration": expiration,

                # TICKERS REAIS
                "sell_ticker": sell["ticker"],
                "buy_ticker": buy["ticker"],

                "sell_strike": sell["strike"],
                "buy_strike": buy["strike"],

                "spot_price": sell["spot_price"],

                "sell_premium": sell["fechamento_d-1"],
                "buy_premium": buy["fechamento_d-1"],
                "credit": credit,

                "sell_volume": sell["volume_d-1"],
                "buy_volume": buy["volume_d-1"],
            })

df_call_spreads = pd.DataFrame(spreads)

print("Qtd spreads gerados:", len(df_call_spreads))
display(df_call_spreads.head(20))

SPOT: 387
CALL: 891
CALLs com spot mapeado: 891


,ticker,fechamento_d-1,volume_d-1,strike,expiration,type,underlying,spot_price
20,ABEVC116,4.29,23000,10.94,2026-03-20,CALL,ABEV3,15.0
21,ABEVC123,3.37,200,11.69,2026-03-20,CALL,ABEV3,15.0
22,ABEVC126,3.13,1000,11.94,2026-03-20,CALL,ABEV3,15.0
23,ABEVC128,2.90,6300,12.19,2026-03-20,CALL,ABEV3,15.0
24,ABEVC131,2.63,5700,12.44,2026-03-20,CALL,ABEV3,15.0
25,ABEVC133,2.39,4400,12.69,2026-03-20,CALL,ABEV3,15.0
26,ABEVC136,2.16,11700,12.94,2026-03-20,CALL,ABEV3,15.0
27,ABEVC138,1.91,16200,13.19,2026-03-20,CALL,ABEV3,15.0
29,ABEVC141,1.72,4200,13.44,2026-03-20,CALL,ABEV3,15.0
30,ABEVC143,1.42,1800,13.69,2026-03-20,CALL,ABEV3,15.0


CALLs OTM: 384


,ticker,underlying,spot_price,strike,expiration,fechamento_d-1,volume_d-1
37,ABEVC158,ABEV3,15.0,15.19,2026-03-20,0.21,107500
39,ABEVC161,ABEV3,15.0,15.44,2026-03-20,0.12,41200
40,ABEVC163,ABEV3,15.0,15.69,2026-03-20,0.07,46900
41,ABEVC166,ABEV3,15.0,15.94,2026-03-20,0.05,19900
42,ABEVC168,ABEV3,15.0,16.19,2026-03-20,0.02,60900
43,ABEVC171,ABEV3,15.0,16.44,2026-03-20,0.02,6100
44,ABEVC173,ABEV3,15.0,16.69,2026-03-20,0.01,6100
45,ABEVC176,ABEV3,15.0,16.94,2026-03-20,0.01,200
61,ABEVD151,ABEV3,15.0,15.17,2026-04-17,0.54,36000
62,ABEVD154,ABEV3,15.0,15.42,2026-04-17,0.39,700


Qtd spreads gerados: 5867


,underlying,expiration,sell_ticker,buy_ticker,sell_strike,buy_strike,spot_price,sell_premium,buy_premium,credit,sell_volume,buy_volume
0,ABEV3,2026-03-20,ABEVC158,ABEVC161,15.19,15.44,15.0,0.21,0.12,0.09,107500,41200
1,ABEV3,2026-03-20,ABEVC158,ABEVC163,15.19,15.69,15.0,0.21,0.07,0.14,107500,46900
2,ABEV3,2026-03-20,ABEVC158,ABEVC166,15.19,15.94,15.0,0.21,0.05,0.16,107500,19900
3,ABEV3,2026-03-20,ABEVC158,ABEVC168,15.19,16.19,15.0,0.21,0.02,0.19,107500,60900
4,ABEV3,2026-03-20,ABEVC158,ABEVC171,15.19,16.44,15.0,0.21,0.02,0.19,107500,6100
5,ABEV3,2026-03-20,ABEVC158,ABEVC173,15.19,16.69,15.0,0.21,0.01,0.20,107500,6100
6,ABEV3,2026-03-20,ABEVC158,ABEVC176,15.19,16.94,15.0,0.21,0.01,0.20,107500,200
7,ABEV3,2026-03-20,ABEVC161,ABEVC163,15.44,15.69,15.0,0.12,0.07,0.05,41200,46900
8,ABEV3,2026-03-20,ABEVC161,ABEVC166,15.44,15.94,15.0,0.12,0.05,0.07,41200,19900
9,ABEV3,2026-03-20,ABEVC161,ABEVC168,15.44,16.19,15.0,0.12,0.02,0.10,41200,60900


In [15]:
# %%
# =========================================
# PARÂMETROS
# =========================================

MIN_SELL_VOLUME    = 5000
MIN_BUY_VOLUME     = 1000
MIN_OTM_PCT        = 2.0
MIN_RETURN_PCT     = 30.0
MIN_CREDIT         = 0.5

MIN_SPREAD_WIDTH   = 1.0
MAX_SPREAD_WIDTH   = 2.0


# =========================================
# MÉTRICAS (IGUAL AO MT5)
# =========================================

df = df_call_spreads.copy()

df["spread_width"] = df["buy_strike"] - df["sell_strike"]

df["otm_pct"] = (
    (df["sell_strike"] - df["spot_price"]) / df["spot_price"]
) * 100

df["max_loss"] = df["spread_width"] - df["credit"]

df["return_pct"] = (df["credit"] / df["max_loss"]) * 100


# =========================================
# FILTROS
# =========================================

df_filt = df[
    (df["sell_volume"]   >= MIN_SELL_VOLUME)   &
    (df["buy_volume"]    >= MIN_BUY_VOLUME)    &
    (df["otm_pct"]       >= MIN_OTM_PCT)       &
    (df["return_pct"]    >= MIN_RETURN_PCT)    &
    (df["credit"]        >= MIN_CREDIT)        &
    (df["spread_width"]  >= MIN_SPREAD_WIDTH)  &
    (df["spread_width"]  <= MAX_SPREAD_WIDTH)  &
    (df["max_loss"]      > 0)
].copy()

print("Qtd travas após filtros:", len(df_filt))


# =========================================
# SEPARA VENCIMENTOS
# =========================================

df_filt["exp_date"] = df_filt["expiration"].dt.date

df_current = df_filt[df_filt["exp_date"] == current_expiry_date]
df_next    = df_filt[df_filt["exp_date"] == next_expiry_date]


# =========================================
# EXIBIÇÃO
# =========================================

def show_table(df_input, titulo):
    print("")
    print("================================")
    print(titulo)
    print("================================")

    if df_input.empty:
        print("Nenhuma trava encontrada")
        return

    df_rank = df_input.sort_values(
        ["return_pct", "otm_pct"],
        ascending=[False, False]
    )

    display(df_rank.head(20))


show_table(df_current, "TRAVAS — VENCIMENTO MÊS ATUAL")
show_table(df_next, "TRAVAS — VENCIMENTO PRÓXIMO MÊS")

Qtd travas após filtros: 40

TRAVAS — VENCIMENTO MÊS ATUAL


,underlying,expiration,sell_ticker,buy_ticker,sell_strike,buy_strike,spot_price,sell_premium,buy_premium,credit,sell_volume,buy_volume,spread_width,otm_pct,max_loss,return_pct,exp_date
1669,PETR4,2026-03-20,PETRC12,PETRC520,49.50,51.15,44.67,1.39,0.08,1.31,31900,584300,1.65,10.812626,0.34,385.294118,2026-03-20
3555,VALE3,2026-03-20,VALEC801,VALEC821,80.18,82.18,78.30,0.89,0.35,0.54,1655300,1334000,2.00,2.401022,1.46,36.986301,2026-03-20



TRAVAS — VENCIMENTO PRÓXIMO MÊS


,underlying,expiration,sell_ticker,buy_ticker,sell_strike,buy_strike,spot_price,sell_premium,buy_premium,credit,sell_volume,buy_volume,spread_width,otm_pct,max_loss,return_pct,exp_date
4348,VALE3,2026-04-17,VALED809,VALED824,80.90,82.40,78.30,2.42,1.80,0.62,477100,124500,1.50,3.320562,0.88,70.454545,2026-04-17
4259,VALE3,2026-04-17,VALED799,VALED814,79.90,81.40,78.30,2.82,2.20,0.62,379500,222000,1.50,2.043423,0.88,70.454545,2026-04-17
4260,VALE3,2026-04-17,VALED799,VALED819,79.90,81.90,78.30,2.82,2.01,0.81,379500,302100,2.00,2.043423,1.19,68.067227,2026-04-17
5557,WEGE3,2026-04-17,WEGED473,WEGED48,47.36,48.86,46.21,1.44,0.88,0.56,8000,22900,1.50,2.488639,0.94,59.574468,2026-04-17
2007,PETR4,2026-04-17,PETRD456,PETRD471,45.65,47.15,44.67,1.90,1.34,0.56,237000,475200,1.50,2.193866,0.94,59.574468,2026-04-17
4305,VALE3,2026-04-17,VALED804,VALED824,80.40,82.40,78.30,2.53,1.80,0.73,432300,124500,2.00,2.681992,1.27,57.480315,2026-04-17
3317,SUZB3,2026-04-17,SUZBD548,SUZBD563,54.88,56.38,53.40,1.52,0.98,0.54,46700,60200,1.50,2.771536,0.96,56.250000,2026-04-17
5558,WEGE3,2026-04-17,WEGED473,WEGED490,47.36,49.11,46.21,1.44,0.83,0.61,8000,35300,1.75,2.488639,1.14,53.508772,2026-04-17
2008,PETR4,2026-04-17,PETRD456,PETRD474,45.65,47.40,44.67,1.90,1.29,0.61,237000,275100,1.75,2.193866,1.14,53.508772,2026-04-17
5582,WEGE3,2026-04-17,WEGED488,WEGED490,47.61,49.11,46.21,1.35,0.83,0.52,7600,35300,1.50,3.029647,0.98,53.061224,2026-04-17


In [16]:
# =========================================
# TELEGRAM
# =========================================

import requests

TELEGRAM_BOT_TOKEN = "8363974523:AAGGmbw7gfIUlCnjkYVSBLKXbiShQhA5yGg"
TELEGRAM_CHAT_ID = "8564196952"


def send_telegram_message(text):

    url = f"https://api.telegram.org/bot{TELEGRAM_BOT_TOKEN}/sendMessage"

    payload = {
        "chat_id": TELEGRAM_CHAT_ID,
        "text": text,
        "parse_mode": "Markdown"
    }

    requests.post(url, json=payload)

In [17]:
def format_spread_message(row):

    msg = f"""
📊 *Trava de Crédito Detectada*

{row.sell_ticker} / {row.buy_ticker}

Spot: {row.spot_price:.2f}

SELL strike: {row.sell_strike:.2f}
BUY strike:  {row.buy_strike:.2f}

Crédito: {row.credit:.2f}
Risco:   {row.max_loss:.2f}

Width: {row.spread_width:.2f}

OTM: {row.otm_pct:.2f}%
Retorno: {row.return_pct:.2f}%
"""

    return msg

In [20]:
# =========================================
# TELEGRAM ALERT — BLOOMBERG STYLE
# =========================================

df_rank_next = df_filt[
    df_filt["exp_date"] == next_expiry_date
].sort_values(
    ["return_pct", "otm_pct"],
    ascending=[False, False]
)

top_spreads = df_rank_next.head(10)

msg = "📊 *TOP 10 TRAVAS - PRÓX VCTO*\n\n"
msg += "`SELL/BUY        OTM   CR   RET`\n"

for row in top_spreads.itertuples():

    sell = row.sell_ticker
    buy  = row.buy_ticker

    msg += (
        f"`{sell}/{buy[-3:]} "
        f"{row.otm_pct:>5.2f} "
        f"{row.credit:>4.2f} "
        f"{row.return_pct:>4.0f}`\n"
    )

send_telegram_message(msg)